# NYC Taxi EDA – Tháng 01/2026
**Nhóm 4 – Tuần 1**  
Đọc từ HDFS Bronze layer, khám phá dữ liệu, phát hiện vấn đề chất lượng.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = (
    SparkSession.builder
    .appName('NYC_Taxi_EDA_Notebook')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

BRONZE = 'hdfs://localhost:9000/nyc_taxi/bronze/raw/yellow_tripdata_2026-01.parquet'
df = spark.read.parquet(BRONZE)
print(f'Rows: {df.count():,}  |  Cols: {len(df.columns)}')

In [ ]:
# Schema
df.printSchema()

In [ ]:
# Thống kê mô tả
df.describe('passenger_count','trip_distance','fare_amount','tip_amount','total_amount').show()

In [ ]:
# NULL counts
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

In [ ]:
# Phân phối theo giờ – biểu đồ
hourly_pd = (
    df.withColumn('hour', F.hour('tpep_pickup_datetime'))
      .groupBy('hour').count()
      .orderBy('hour')
      .toPandas()
)

plt.figure(figsize=(12, 4))
sns.barplot(data=hourly_pd, x='hour', y='count', color='steelblue')
plt.title('Số chuyến theo giờ trong ngày – NYC Taxi 01/2026')
plt.xlabel('Giờ'); plt.ylabel('Số chuyến')
plt.tight_layout(); plt.show()

In [ ]:
# Phân phối fare_amount
fare_pd = df.select('fare_amount').filter(F.col('fare_amount').between(0, 100)).toPandas()

plt.figure(figsize=(10, 4))
plt.hist(fare_pd['fare_amount'], bins=50, color='coral', edgecolor='white')
plt.title('Phân phối fare_amount (0–100$)')
plt.xlabel('Fare ($)'); plt.ylabel('Tần suất')
plt.tight_layout(); plt.show()

In [ ]:
spark.stop()
print('Done')